# 01 - Environment and dataset exploration

In [ ]:
# install packages and pull the repo
!pip install -q timm albumentations onnx onnxruntime
!git clone -q https://github.com/YOUR-USERNAME/dr-triage.git /kaggle/working/dr-triage 2>/dev/null || git -C /kaggle/working/dr-triage pull -q
%cd /kaggle/working/dr-triage

In [ ]:
# imports
import sys, os, json, time, random, warnings
from pathlib import Path

sys.path.insert(0, '/kaggle/working/dr-triage')
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cv2

# output folders
for d in ['results/figures', 'results/tables', 'results/logs', 'models/export']:
    Path(d).mkdir(parents=True, exist_ok=True)

In [ ]:
# check accelerator
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | {props.total_memory / 1e9:.1f} GB')
else:
    print('GPU: NONE')

In [ ]:
# reproducibility
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()

# config
from src.config import Config

cfg = Config()
cfg.checkpoint_dir = '/kaggle/working/checkpoints'
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.num_workers = 2

# dataset paths
DDR_ROOT   = '/kaggle/input/ddrdataset'
APTOS_ROOT = '/kaggle/input/aptos2019-blindness-detection'

In [ ]:
# show config
pd.Series(cfg.to_dict()).to_frame('value')

In [ ]:
# list mounted datasets
!ls /kaggle/input/

In [ ]:
# inspect DDR folder layout
from src.data_sources import inspect_layout

inspect_layout(DDR_ROOT)

In [ ]:
# DDR manifest
from src.data_sources import load_ddr

ddr = load_ddr(DDR_ROOT)
print('split_source:', ddr['split_source'].iloc[0])
ddr.head()

In [ ]:
# APTOS manifest (external set)
from src.data_sources import load_aptos

aptos = load_aptos(APTOS_ROOT)
print(f'APTOS: {len(aptos)} images')
aptos.head(3)

In [ ]:
# combined manifest
manifest = pd.concat([ddr, aptos], ignore_index=True)
manifest.to_csv('results/tables/manifest.csv', index=False)
manifest.groupby(['source', 'split']).size().to_frame('images')

In [ ]:
# class distribution table
from src.data_sources import imbalance_report

dist = imbalance_report(ddr)
dist.to_csv('results/tables/table1_class_distribution.csv', index=False)
dist

In [ ]:
# class distribution plot
fig, ax = plt.subplots(figsize=(7.5, 3.6))
bars = ax.bar(dist['name'], dist['count'], color='#7F77DD', edgecolor='none')
ax.set_yscale('log')
ax.set_ylabel('images (log scale)')
ax.set_title('DDR grade distribution')
for b, c, p in zip(bars, dist['count'], dist['percent']):
    ax.text(b.get_x() + b.get_width()/2, c, f'{c}\n{p}%',
            ha='center', va='bottom', fontsize=8)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('results/figures/fig2_class_distribution.png', dpi=200)
plt.show()

In [ ]:
# ungradable image counts
n_ungradable = int(ddr['ungradable'].sum())
n_total = len(ddr)
print(f'ungradable: {n_ungradable} / {n_total} = {n_ungradable/n_total:.1%}')
print(ddr.groupby('split')['ungradable'].agg(['sum', 'mean']).rename(
    columns={'sum': 'ungradable', 'mean': 'fraction'}))

In [ ]:
# split summary table
from src.data_sources import describe

summary = describe(manifest)
summary.to_csv('results/tables/table2_split_summary.csv')
summary

In [ ]:
# image size statistics on a sample
sample_paths = ddr.sample(60, random_state=SEED)['image']
shapes = []
for rel in sample_paths:
    img = cv2.imread(f'{DDR_ROOT}/{rel}')
    if img is not None:
        h, w = img.shape[:2]
        shapes.append({'h': h, 'w': w, 'aspect': round(w/h, 3), 'mp': round(h*w/1e6, 2)})

shapes = pd.DataFrame(shapes)
shapes.describe().round(2)

In [ ]:
# example images per grade
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for col, grade in enumerate(range(5)):
    subset = ddr[(ddr['level'] == grade)].sample(3, random_state=SEED)
    for row, (_, r) in enumerate(subset.iterrows()):
        img = cv2.imread(f"{DDR_ROOT}/{r['image']}")
        axes[row, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f'Grade {grade}', fontsize=11)
plt.suptitle('DDR examples by DR grade', y=0.98)
plt.tight_layout()
plt.savefig('results/figures/fig3_samples_by_grade.png', dpi=150)
plt.show()

In [ ]:
# ungradable examples
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, (_, r) in zip(axes, ddr[ddr['ungradable']].sample(5, random_state=SEED).iterrows()):
    img = cv2.imread(f"{DDR_ROOT}/{r['image']}")
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.suptitle('DDR ungradable images')
plt.tight_layout()
plt.savefig('results/figures/fig3b_ungradable_samples.png', dpi=150)
plt.show()